# LexVerse 使用教程

本教程通过一组可直接运行的示例，介绍本地法律语料库的检索、读取与 Provider 集成。

## 数据集合

- **legal_laws**：法律法规、司法解释（74,642 条）
- **legal_cases**：法律案件、判决书（150,135 条）
- **legal_qa**：法律问答（349 条）
- **legal_concepts**：法律概念、术语（56,136 条）
- **legal_templates**：文书模板（139 条）

## 1. 环境初始化

In [3]:
import os
import json
from pathlib import Path
from lexverse_env import CorpusEnv

# 设置路径（在 examples 目录下运行时）
data_dir = Path("../data")
state_dir = Path("../.lexverse")

print(f"数据目录: {data_dir}")
print(f"索引目录: {state_dir}")
print(f"数据目录存在: {data_dir.exists()}")
print(f"索引目录存在: {state_dir.exists()}")

# 检查索引状态
if state_dir.exists():
    index_file = state_dir / "catalog.sqlite"
    if index_file.exists():
        size_mb = index_file.stat().st_size / (1024 * 1024)
        print(f"\n索引状态: ✅ 已构建")
        print(f"索引大小: {size_mb:.1f} MB")
    else:
        print(f"\n索引状态: ❌ 未构建")
        print("请运行: lexverse-env build-index --data-dir ../data --state-dir ../.lexverse")
else:
    print(f"\n索引状态: ❌ 索引目录不存在")

数据目录: ../data
索引目录: ../.lexverse
数据目录存在: True
索引目录存在: True

索引状态: ✅ 已构建
索引大小: 13782.8 MB


## 2. 获取环境清单

In [30]:
# 打开环境并读取清单
with CorpusEnv.open(data_dir, state_dir) as env:
    manifest = env.get_manifest()
    print("\n【索引信息】")
    print(f"快照ID: {manifest['snapshot_id']}")
    print(f"索引版本: {manifest['index_generation']}")
    print(f"可用 Providers: {manifest['providers_available']}")
    print(f"\n集合统计:")
    for col_name, col_info in manifest.get('collections', {}).items():
        print(f"  {col_name}: {col_info.get('record_count', 0)} 条记录")



【索引信息】
快照ID: stat-981d5f119605451d51e1c167
索引版本: ebe9b6b82367900f88dd265c74bfaa82
可用 Providers: ['local']

集合统计:
  legal_cases: 150135 条记录
  legal_concepts: 56136 条记录
  legal_laws: 74642 条记录
  legal_qa: 349 条记录
  legal_templates: 139 条记录


## 3. 列出所有集合

In [31]:
with CorpusEnv.open(data_dir, state_dir) as env:
    collections = env.list_collections()
    print("\n【可用集合】")
    for col in collections['collections']:
        print(f"\n{col['name']} ({col['label']})")
        print(f"  记录数: {col['record_count']}")
        print(f"  Providers: {col['providers']}")
        print(f"  功能: {col['capabilities']}")



【可用集合】

legal_cases (法律案件)
  记录数: 150135
  Providers: ['local']
  功能: ['search', 'get', 'read_part']

legal_concepts (法律概念)
  记录数: 56136
  Providers: ['local']
  功能: ['search', 'get', 'read_part']

legal_laws (法律法规)
  记录数: 74642
  Providers: ['local']
  功能: ['search', 'get', 'read_part']

legal_qa (法律问答)
  记录数: 349
  Providers: ['local']
  功能: ['search', 'get', 'read_part']

legal_templates (文书模板)
  记录数: 139
  Providers: ['local']
  功能: ['search', 'get', 'read_part', 'asset']


## 4. 查看集合详情

In [32]:
with CorpusEnv.open(data_dir, state_dir) as env:
    # 查看案例集合的字段与能力
    desc = env.describe_collection("legal_cases")
    print("\n【legal_cases 集合详情】")
    print(json.dumps(desc, ensure_ascii=False, indent=2))



【legal_cases 集合详情】
{
  "collection": "legal_cases",
  "provider": "local",
  "sources": [
    "agentscourt",
    "j1bench",
    "judge",
    "lecardv2",
    "legal_world",
    "lexchain",
    "maser",
    "mslr_bench",
    "muser"
  ],
  "filter_fields": [
    "source",
    "dataset",
    "category",
    "case_type",
    "court",
    "stage",
    "year"
  ],
  "search_fields": [
    "title",
    "case_number",
    "case_cause",
    "court",
    "facts"
  ],
  "capabilities": [
    "search",
    "get",
    "read_part"
  ],
  "raw_schema": "source-specific"
}


## 5. 基础搜索：危险驾驶案例

In [33]:
with CorpusEnv.open(data_dir, state_dir) as env:
    # 搜索危险驾驶案例
    page = env.search_records(
        collection="legal_cases",
        query="危险驾驶",
        limit=3
    )

    print("\n【搜索结果】")
    print(f"找到 {len(page['items'])} 条记录")
    print(f"是否有更多: {'有' if page['next_cursor'] else '没有'}")

    for i, item in enumerate(page['items'], 1):
        print(f"\n[{i}] ID: {item['id']}")
        if 'key_fields' in item:
            kf = item['key_fields']
            print(f"    标题: {kf.get('title', 'N/A')}")
            print(f"    案号: {kf.get('case_number', 'N/A')}")
            print(f"    案由: {kf.get('case_cause', 'N/A')}")
            print(f"    法院: {kf.get('court', 'N/A')}")



【搜索结果】
找到 3 条记录
是否有更多: 有

[1] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(2).json::ordinal-48
    标题: 韩灵龙危险驾驶罪刑事二审刑事判决书
    案号: （2022）粤18刑终4号
    案由: 危险驾驶
    法院: 广东省清远市中级人民法院

[2] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(1).json::ordinal-12
    标题: 董峰危险驾驶刑事一审刑事判决书
    案号: （2023）粤0113刑初1253号
    案由: 危险驾驶
    法院: 广东省广州市番禺区人民法院

[3] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(2).json::ordinal-5
    标题: 王颖危险驾驶罪刑事二审刑事判决书
    案号: （2023）吉01刑终35号
    案由: 危险驾驶
    法院: 吉林省长春市中级人民法院


## 6. 读取完整记录

In [34]:
with CorpusEnv.open(data_dir, state_dir) as env:
    # 搜索并读取第一条记录
    page = env.search_records(
        collection="legal_cases",
        query="危险驾驶",
        limit=1
    )

    if page['items']:
        record_id = page['items'][0]['id']
        record = env.get_record(record_id)

        print("\n【完整记录】")
        print(f"ID: {record['id']}")
        print(f"\n来源信息:")
        print(json.dumps(record['provenance'], ensure_ascii=False, indent=2))
        print(f"\n记录内容（前1000字符）:")
        print(json.dumps(record['record'], ensure_ascii=False, indent=2)[:1000] + "...")



【完整记录】
ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(2).json::ordinal-48

来源信息:
{
  "provider": "local",
  "collection": "legal_cases",
  "source": "agentscourt",
  "source_key": "legal_cases/agentscourt/SimuCourt(2).json::ordinal-48",
  "snapshot_id": "stat-981d5f119605451d51e1c167"
}

记录内容（前1000字符）:
{
  "序号": 48,
  "案件名": "韩灵龙危险驾驶罪刑事二审刑事判决书",
  "类别": "刑事",
  "法院": "广东省清远市中级人民法院",
  "案由": "危险驾驶",
  "案号": "（2022）粤18刑终4号",
  "发布时间": "2022-05-21\t",
  "审理程序": "二审",
  "上诉人": "（原审被告人）韩灵龙",
  "上诉人基本情况": "无",
  "上诉人辩护": "上诉人韩灵龙提出，上诉人驾驶前饮用了半杯水酒和三杯啤酒属于酒驾，并非危险驾驶，并且不是被交警当场抓获，228mg/100mL酒精含量是停好车辆后又喝酒的测试结果，请求法院重审改判为酒驾而非危险驾驶。",
  "被上诉人": "检察院",
  "被上诉人辩护": "原审认定事实清楚，证据确实、充分，适用法律正确，定罪量刑准确，上诉人的上诉理由无事实与法律依据，建议二审维持原判",
  "上诉请求": "连州市人民法院审理连州市人民检察院指控原审被告人韩灵龙犯危险驾驶罪一案，于2021年12月6日作出（2021）粤1882刑初245号刑事判决。原审被告人韩灵龙不服，提出上诉。上诉人韩灵龙提出，上诉人驾驶前饮用了半杯水酒和三杯啤酒属于酒驾，并非危险驾驶，并且不是被交警当场抓获，228mg/100mL酒精含量是停好车辆后又喝酒的测试结果，请求法院重审改判为酒驾而非危险驾驶。",
  "一审法院认定事实": "原判认定：2021年7月14日15时50分许，被告人韩灵龙驾驶粤Ｒ8**号小型轿车搭载刘某洲、韦某志、韦某来、韦某文从X

## 7. 过滤搜索：2023年的劳动合同案件

In [35]:
with CorpusEnv.open(data_dir, state_dir) as env:
    page = env.search_records(
        collection="legal_cases",
        query="劳动合同",
        filters={"year": "2023"},
        limit=5
    )

    print("\n【2023 年劳动合同案件】")
    print(f"找到 {len(page['items'])} 条记录\n")

    for i, item in enumerate(page['items'], 1):
        kf = item.get('key_fields', {})
        print(f"[{i}] {kf.get('title', 'N/A')}")
        print(f"    案号: {kf.get('case_number', 'N/A')}")



【2023 年劳动合同案件】
找到 5 条记录

[1] 詹志强、江鹏飞劳务合同纠纷民事一审民事判决书
    案号: （2023）粤0113民初18134号
[2] 徐新春、宁波甬泉机械制造有限公司劳动争议一审民事判决书
    案号: （2023）浙0203民初7559号
[3] 宁波远东汽车部件制造有限公司、何小琴劳动争议一审民事判决书
    案号: （2023）浙0203民初7024号
[4] 付桃桃、凉城县人力资源和社会保障局行政一审行政判决书
    案号: （2023）内0925行初2号
[5] 刘庆与沈阳市和平区人力资源和社会保障局、田斌其他一审行政判决书
    案号: （2023）辽0103行初79号


## 8. 搜索法律条文

In [58]:
with CorpusEnv.open(data_dir, state_dir) as env:
    page = env.search_records(
        collection="legal_laws",
        query="劳动合同法",
        limit=5
    )

    print("\n【劳动合同法相关条文】")
    for i, item in enumerate(page['items'], 1):
        kf = item.get('key_fields', {})
        print(f"\n[{i}] {kf.get('law_name', 'N/A')}")
        print(f"    条文: {kf.get('article', 'N/A')}")
        print(f"    类别: {kf.get('category', 'N/A')}")
        print(f"    生效日期: {kf.get('effective_from', 'N/A')}")



【劳动合同法相关条文】

[1] 中华人民共和国劳动合同法实施条例
    条文: 第六条
    类别: 行政法规
    生效日期: 2008-09-18

[2] 中华人民共和国劳动合同法实施条例
    条文: 第十九条
    类别: 行政法规
    生效日期: 2008-09-18

[3] 中华人民共和国劳动合同法实施条例
    条文: 第十八条
    类别: 行政法规
    生效日期: 2008-09-18

[4] 中华人民共和国劳动合同法实施条例
    条文: 第十一条
    类别: 行政法规
    生效日期: 2008-09-18

[5] 中华人民共和国劳动合同法实施条例
    条文: 第二十六条
    类别: 行政法规
    生效日期: 2008-09-18


## 9. 搜索法律问答

In [59]:
with CorpusEnv.open(data_dir, state_dir) as env:
    page = env.search_records(
        collection="legal_qa",
        query="劳动合同解除",
        limit=3
    )

    print("\n【劳动合同解除问答】")
    for i, item in enumerate(page['items'], 1):
        record = env.get_record(item['id'])
        content = record['record']

        # 兼容不同数据格式
        question = None
        answer = None

        # 格式 1: 标准 question/answer（fadawang, dlawbench）
        if 'question' in content:
            question = content.get('question')
            answer = content.get('answer', 'N/A')

        # 格式 2: theme + topic_list（j1bench）
        elif 'theme' in content and 'topic_list' in content:
            question = content.get('theme')
            topics = content.get('topic_list', [])
            if topics:
                topic_parts = []
                for t in topics[:3]:  # 最多显示 3 个子问题
                    topic_parts.append(f"问：{t.get('topic')} 答：{t.get('content')}")
                answer = ' | '.join(topic_parts)
            else:
                answer = 'N/A'

        # 格式 3: topic_name + content
        elif 'topic_name' in content:
            question = content.get('topic_name')
            answer = content.get('content', 'N/A')

        else:
            question = 'N/A'
            answer = 'N/A'

        print(f"\n[{i}] 问题: {question}")
        if len(str(answer)) > 200:
            print(f"    回答: {answer[:200]}...")
        else:
            print(f"    回答: {answer}")


【劳动合同解除问答】

[1] 问题: 用人单位与劳动者连续两次订立固定期限劳动合同，期满后用人单位对续订无固定期限劳动合同是否享有选择权？
    回答: 劳动合同法第十四条第二款规定：“用人单位与劳动者协商一致，可以订立无固定期限劳动合同。有下列情形之一，劳动者提出或者同意续订、订立劳动合同的，除劳动者提出订立固定期限劳动合同外，应当订立无固定期限劳动合同：……（三）连续订立二次固定期限劳动合同，且劳动者没有本法第三十九条和第四十条第一项、第二项规定的情形，续订劳动合同的。”根据该规定，在劳动者不存在劳动合同法第三十九条和第四十条第一项、第二项规定...

[2] 问题: 当事人起诉主张解除合同后撤诉，又再次起诉主张解除合同，合同解除时间如何确定?原告起诉解除合同，后变更诉讼请求为继续履行，被告主张合同在起诉状副本送达时解除，应否支持?
    回答: 第一，对于当事人起诉主张解除合同后撤诉，又再次起诉主张解除合同时，合同解除时间如何确定的问题，应当体系化适用民法典第五百六十五条和《民法典合同编通则解释》第五十四条的规定。具体而言：
一是依照民法典第五百六十五条第二款的规定，合同自起诉状副本送达对方时解除，其前提是人民法院确认当事人解除合同的主张。因此，即使当事人一方请求解除合同的起诉状副本已经到达对方当事人，在未经人民法院审理确认其主张之前，并...

[3] 问题: 劳动合同书面形式与法律责任
    回答: 问：事实劳动关系是否需要订立书面劳动合同才能成立？ 答：否 | 问：用人单位与劳动者未订立书面劳动合同是否影响劳动争议的受理？ 答：否 | 问：用人单位自用工之日起一个月内未订立书面劳动合同是否有法律责任？ 答：是


## 10. 搜索法律概念

In [60]:
with CorpusEnv.open(data_dir, state_dir) as env:
    page = env.search_records(
        collection="legal_concepts",
        query="不动产登记",
        limit=3
    )

    print("\n【不动产登记相关概念】")
    for i, item in enumerate(page['items'], 1):
        record = env.get_record(item['id'])
        content = record['record']

        # 兼容不同数据格式
        term = None
        concept_text = None
        category = None
        domain = None

        # 格式 1: chinese_legal_terms（term, concept, category, domain）
        if 'term' in content:
            term = content.get('term')
            concept_text = content.get('concept', 'N/A')
            category = content.get('category', 'N/A')
            domain = content.get('domain', 'N/A')

        # 格式 2: ownthink（entity_name, facts）
        elif 'entity_name' in content:
            term = content.get('entity_name')
            facts = content.get('facts', {})
            # 从 facts 中提取信息
            if isinstance(facts, dict):
                concept_text = str(facts)
                tags = facts.get('标签', [])
                category = ', '.join(tags) if tags else 'N/A'
            else:
                concept_text = 'N/A'
                category = 'N/A'
            domain = 'N/A'

        else:
            term = 'N/A'
            concept_text = 'N/A'
            category = 'N/A'
            domain = 'N/A'

        print(f"\n[{i}] 术语: {term}")
        print(f"    类别: {category}")
        if domain != 'N/A':
            print(f"    领域: {domain}")
        if len(str(concept_text)) > 100:
            print(f"    定义: {concept_text[:100]}...")
        else:
            print(f"    定义: {concept_text}")


【不动产登记相关概念】

[1] 术语: 不动产登记
    类别: 物权
    领域: 民法
    定义: 不动产登记是指不动产登记机构依法将不动产权利归属和其他法定事项记载于不动产登记簿的行为。此处的不动产是指土地、海域，以及房屋、森林、林木等建筑物、构筑物和定着物。
我国对不动产实行统一的登记制度。不动...

[2] 术语: 不动产登记与物权法以登记为中心
    类别: 法律书籍
    定义: {'标签': ['法律书籍']}

[3] 术语: 不动产登记簿
    类别: 法律
    定义: {'标签': ['法律']}


## 11. 搜索文书模板

In [62]:
with CorpusEnv.open(data_dir, state_dir) as env:
    page = env.search_records(
        collection="legal_templates",
        query="合同",
        limit=3
    )

    print("\n【合同相关文书模板】")
    for i, item in enumerate(page['items'], 1):
        kf = item.get('key_fields', {})
        print(f"\n[{i}] {kf.get('title', 'N/A')}")
        
        # 读取完整记录获取文件路径
        record = env.get_record(item['id'])
        file_path = record['record'].get('file_path', 'N/A')
        print(f"    文件: {file_path}")


【合同相关文书模板】

[1] 融资租赁合同纠纷民事起诉状（2025版）
    文件: files/融资租赁合同纠纷民事起诉状（2025版）.doc

[2] 房屋买卖合同纠纷民事起诉状
    文件: files/房屋买卖合同纠纷民事起诉状.docx

[3] 买卖合同纠纷民事起诉状
    文件: files/买卖合同纠纷民事起诉状.docx


## 12. 分页查询

In [40]:
with CorpusEnv.open(data_dir, state_dir) as env:
    # 查询第一页
    page1 = env.search_records(
        collection="legal_cases",
        query="合同纠纷",
        limit=3
    )

    print("\n【第一页】")
    for item in page1['items']:
        print(f"  {item.get('key_fields', {}).get('title', 'N/A')}")

    # 使用游标查询下一页
    if page1['next_cursor']:
        page2 = env.search_records(
            collection="legal_cases",
            query="合同纠纷",
            limit=3,
            cursor=page1['next_cursor']
        )

        print("\n【第二页】")
        for item in page2['items']:
            print(f"  {item.get('key_fields', {}).get('title', 'N/A')}")



【第一页】
  张某某、环县魏家河鑫磊石料厂借款合同纠纷民事二审民事判决书
  四川春航建设集团有限公司、青海博友建材有限公司买卖合同纠纷民事二审民事判决书
  詹志强、江鹏飞劳务合同纠纷民事一审民事判决书

【第二页】
  岫岩某酒店、姜某生等买卖合同纠纷民事二审民事判决书
  王文举、郭春平车辆租赁合同纠纷一审民事判决书
  马志豪、浙江华得方商贸有限公司房屋租赁合同纠纷一审民事判决书


## 13. 多集合查询（并行）

In [61]:
with CorpusEnv.open(data_dir, state_dir) as env:
    query = "劳动合同"

    # 查询案例、问答和法律条文
    results = {}
    for collection in ["legal_cases", "legal_qa", "legal_laws"]:
        page = env.search_records(
            collection=collection,
            query=query,
            limit=3
        )
        results[collection] = page['items']

    # 汇总显示结果
    print(f"\n【{query}】多集合查询结果")

    print(f"\n案例 ({len(results['legal_cases'])} 条):")
    for item in results['legal_cases']:
        print(f"  - {item.get('key_fields', {}).get('title', 'N/A')}")

    print(f"\n问答 ({len(results['legal_qa'])} 条):")
    for item in results['legal_qa']:
        # question 可能在 key_fields 中，也可能是 theme
        kf = item.get('key_fields', {})
        q = kf.get('question') or kf.get('theme') or kf.get('id', 'N/A')
        print(f"  - {q}")

    print(f"\n法律条文 ({len(results['legal_laws'])} 条):")
    for item in results['legal_laws']:
        kf = item.get('key_fields', {})
        print(f"  - {kf.get('law_name', 'N/A')} {kf.get('article', '')}")


【劳动合同】多集合查询结果

案例 (3 条):
  - 内蒙古泰伊源机械制造有限公司、固阳县人力资源和社会保障局等行政一审行政判决书
  - 莫铭昌、李海舟劳务合同纠纷民事一审民事判决书
  - 詹志强、江鹏飞劳务合同纠纷民事一审民事判决书

问答 (3 条):
  - 用人单位与劳动者连续两次订立固定期限劳动合同，期满后用人单位对续订无固定期限劳动合同是否享有选择权？
  - 劳动合同书面形式与法律责任
  - 劳动合同中的竞业限制和损失赔偿问题

法律条文 (3 条):
  - 中华人民共和国劳动合同法 第十四条
  - 中华人民共和国劳动合同法实施条例 第十九条
  - 中华人民共和国劳动合同法实施条例 第十八条


## 14. 稳定排序（非相关性排序）

In [44]:
with CorpusEnv.open(data_dir, state_dir) as env:
    # 使用稳定排序，确保结果顺序稳定
    page = env.search_records(
        collection="legal_cases",
        query="合同",
        sort="stable",  # 默认是 "relevance"
        limit=5
    )

    print("\n【稳定排序结果】")
    for i, item in enumerate(page['items'], 1):
        print(f"[{i}] ID: {item['id']}")
        print(f"    标题: {item.get('key_fields', {}).get('title', 'N/A')}")



【稳定排序结果】
[1] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(1).json::ordinal-10
    标题: 万冲侵犯公民个人信息一审刑事判决书
[2] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(1).json::ordinal-100
    标题: 童荥杰盗窃罪一审刑事判决书
[3] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(1).json::ordinal-101
    标题: 刘贵江走私、贩卖、运输、制造毒品罪一审刑事判决书
[4] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(1).json::ordinal-102
    标题: 陈锦、东检刑走私国家禁止进出口的货物、物品罪等刑事一审刑事判决书
[5] ID: lv:case:agentscourt:legal_cases/agentscourt/SimuCourt(1).json::ordinal-103
    标题: 姜锦哲走私国家禁止进出口的货物罪一审刑事判决书


## 15. 集成北大法宝 MCP

`CorpusEnv` 已内置北大法宝的鉴权、MCP 响应解析、服务路由和参数转换。运行本节前，请在启动 Jupyter 的终端设置 `PKULAW_TOKEN`：

```bash
export PKULAW_TOKEN='Bearer <your-token>'
```

统一的 `env.search_records()` 接口支持法规语义、法规关键词、案例语义和案例关键词四种检索模式。

In [ ]:
import os

if not os.environ.get("PKULAW_TOKEN"):
    raise RuntimeError(
        "缺少 PKULAW_TOKEN；请在启动 Jupyter 前设置 "
        "export PKULAW_TOKEN='Bearer <your-token>'"
    )

print("已检测到 PKULAW_TOKEN，CorpusEnv 将自动启用北大法宝 provider。")


已检测到 PKULAW_TOKEN，CorpusEnv 将自动启用北大法宝 provider。


### 四种北大法宝检索模式

不传 `search_mode` 时使用语义检索；设置 `filters={"search_mode": "keyword"}` 时切换到关键词检索。集合名称决定检索法规还是案例。

In [4]:
with CorpusEnv.open(data_dir, state_dir) as env:
    pkulaw_results = {
        "法规语义 / search_article": env.search_records(
            "legal_laws", provider="pkulaw", query="劳动合同解除", limit=2
        ),
        "法规关键词 / get_law_list": env.search_records(
            "legal_laws", provider="pkulaw", query="劳动合同解除",
            filters={"search_mode": "keyword"}, limit=2,
        ),
        "案例语义 / search_case": env.search_records(
            "legal_cases", provider="pkulaw", query="劳动合同纠纷", limit=2
        ),
        "案例关键词 / get_case_list": env.search_records(
            "legal_cases", provider="pkulaw", query="劳动合同纠纷",
            filters={"search_mode": "keyword"}, limit=2,
        ),
    }

for route, page in pkulaw_results.items():
    print(f"\n【{route}】{len(page['items'])} 条")
    for item in page["items"]:
        title = item.get("key_fields", {}).get("title", item["id"])
        print(f"- {title}")



【法规语义 / search_article】2 条
- 劳动部办公厅关于劳动者解除劳动合同有关问题的复函
- 中华人民共和国劳动合同法

【法规关键词 / get_law_list】0 条

【案例语义 / search_case】2 条
- 桂建群与广发证券股份有限公司深圳民田路证券营业部等劳动合同纠纷申请案
- 濮阳市合众石油科技开发有限公司;孙少宁劳动合同纠纷一审民事判决书

【案例关键词 / get_case_list】2 条
- 劳动合同纠纷劳动合同纠纷二审民事判决书
- 劳动合同纠纷劳动合同纠纷二审民事判决书


### 混合查询：本地 + 北大法宝

本地数据和北大法宝使用相同的返回结构，只需切换 `provider`，因此可以直接组合结果。

In [5]:
query = "劳动合同纠纷"
with CorpusEnv.open(data_dir, state_dir) as env:
    local_results = env.search_records(
        "legal_cases", provider="local", query=query, limit=5
    )
    remote_results = env.search_records(
        "legal_cases", provider="pkulaw", query=query, limit=5
    )

combined_items = local_results["items"] + remote_results["items"]
print(f"【混合查询结果：{query}】")
print(f"本地索引: {len(local_results['items'])} 条")
print(f"北大法宝: {len(remote_results['items'])} 条")
print(f"合计: {len(combined_items)} 条")


【混合查询结果：劳动合同纠纷】
本地索引: 5 条
北大法宝: 5 条
合计: 10 条
